# 03 — Transfer Learning (Experiments A1 & A2)

**ICS555 Capstone: African Landmark Recognition**

| Run | Model | Strategy | Expected Top-1 |
|-----|-------|----------|----------------|
| A1  | ResNet-18 | head-only frozen | ~72% |
| A2  | ResNet-18 | full fine-tune   | ~75% est. |

In [ ]:
import os
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    !git clone https://github.com/ajegetina/african-landmark-recognition.git
    %cd african-landmark-recognition
    !pip install -q -r requirements.txt
    from google.colab import drive
    drive.mount('/content/drive')
    !ln -sf /content/drive/MyDrive/landmark_images landmark_images

    import wandb
    wandb.login()

In [ ]:
%matplotlib inline
import sys, torch
sys.path.insert(0, '..')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 1. Data loaders

In [ ]:
from src.data import get_data_loaders
data_loaders = get_data_loaders(batch_size=64, valid_size=0.2, num_workers=2)

## 2. Run A1 — ResNet-18 head-only

Only the final FC layer (25,650 params) is updated. Backbone stays frozen.

In [ ]:
!python ablation_runner.py --config configs/A1_resnet18_frozen.yaml

## 3. Run A2 — ResNet-18 full fine-tune

All 11.2M parameters are trained with a low LR (1e-4). Tests the ablation:
_Does unfreezing the backbone improve fine-grained recognition with our limited data?_

In [ ]:
!python ablation_runner.py --config configs/A2_resnet18_full.yaml

## 4. Compare A1 vs A2

In [ ]:
import torch
from src.transfer import get_model_transfer_learning
from src.optimization import get_loss
from src.train import one_epoch_test
from sklearn.metrics import f1_score

def evaluate(ckpt_path, model_name='resnet18', strategy='full'):
    model = get_model_transfer_learning(model_name, n_classes=50, finetune_strategy=strategy)
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    loss_fn = get_loss()
    _, top1, top5, targets, preds = one_epoch_test(data_loaders['test'], model, loss_fn)
    mf1 = f1_score(targets, preds, average='macro', zero_division=0)
    return top1, top5, mf1

runs = {
    'A1 (frozen)': ('checkpoints/A1_resnet18_frozen.pt', 'resnet18', 'frozen'),
    'A2 (full)':   ('checkpoints/A2_resnet18_full.pt',   'resnet18', 'full'),
}

print(f'{"Run":<20} {"Top-1":>6} {"Top-5":>6} {"Macro-F1":>10}')
print('-' * 46)
for name, (ckpt, model_name, strategy) in runs.items():
    top1, top5, mf1 = evaluate(ckpt, model_name, strategy)
    print(f'{name:<20} {100*top1:5.1f}% {100*top5:5.1f}% {100*mf1:9.1f}%')

## 5. Export A1 as TorchScript

In [ ]:
from src.predictor import Predictor
from src.helpers import compute_mean_and_std

class_names = data_loaders['train'].dataset.classes
mean, std   = compute_mean_and_std()

model_a1 = get_model_transfer_learning('resnet18', n_classes=50, finetune_strategy='frozen').cpu()
model_a1.load_state_dict(torch.load('checkpoints/A1_resnet18_frozen.pt', map_location='cpu'))

predictor = Predictor(model_a1, class_names, mean, std).cpu()
scripted  = torch.jit.script(predictor)
scripted.save('checkpoints/A1_resnet18_frozen_exported.pt')
print('Saved A1 TorchScript checkpoint.')